# Enhanced DQN for Connect-4 (Q4)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Stiles-Clements1/connect4-rl-arena/blob/main/notebooks/colab_simple_dqn.ipynb)

Self-contained DQN trainer. Runs on **Google Colab** (click the badge above,
then: Runtime → Change runtime type → GPU) or **locally** (Jupyter / VS Code /
Cursor, from inside the cloned repo) with no code changes.

**What this notebook does**

1. Sets up a repo-rooted working directory (`REPO_ROOT`, aliased as `COLAB_ROOT`
   for back-compatibility with existing code below).
2. Defines a DQN network with a two-channel `(6, 7, 2)` board input.
3. Trains it against a progressive-difficulty opponent system.
4. Saves the final model to `checkpoints/enhanced_dqn_optimized.h5` and the
   training log to `logs/enhanced_dqn_optimized_log.json`.

To promote a finished DQN for the tournament, copy the file from `checkpoints/`
into `RL models/enhanced_dqn_optimized.h5` — that folder is the canonical
home for deployable agents.

In [ ]:
# Cell 1 — Environment setup (Colab OR local, no edits needed).
#
# Detects whether we're running in Google Colab. On Colab, clones the repo
# (or pulls latest main if it's already there). On local runs, walks up
# from the notebook's working directory to find the repo root. Either way,
# REPO_ROOT points at the repo, src/ is added to sys.path, and the working
# directory is set to the repo so relative paths like 'logs/' and
# 'checkpoints/' resolve correctly.

import os
import sys
import subprocess
from pathlib import Path

import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, models, optimizers
import random
import json
import time
from collections import deque
from typing import List, Tuple, Dict, Any
import matplotlib.pyplot as plt
import warnings

# Quieter TensorFlow logging
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'
tf.get_logger().setLevel('ERROR')

IS_COLAB = 'google.colab' in sys.modules
REPO_URL = 'https://github.com/Stiles-Clements1/connect4-rl-arena.git'

if IS_COLAB:
    # Colab: clone (or pull) the repo into /content/connect4-rl-arena.
    REPO_ROOT = Path('/content/connect4-rl-arena')
    if not REPO_ROOT.exists():
        print(f'Cloning {REPO_URL} → {REPO_ROOT}')
        subprocess.run(['git', 'clone', REPO_URL, str(REPO_ROOT)], check=True)
    else:
        print(f'Repo already at {REPO_ROOT}; pulling latest main.')
        subprocess.run(['git', '-C', str(REPO_ROOT), 'pull', '--ff-only'], check=False)
else:
    # Local: walk up from the notebook's CWD until we hit the repo sentinel.
    here = Path.cwd().resolve()
    REPO_ROOT = None
    for p in [here, *here.parents]:
        if (p / 'src' / 'game_engine.py').exists():
            REPO_ROOT = p
            break
    if REPO_ROOT is None:
        raise RuntimeError(
            f'Could not find repo root starting from {here}. '
            f'Run the notebook from inside the cloned connect4-rl-arena repo.'
        )
    print(f'Local repo root: {REPO_ROOT}')

# Back-compat alias for existing cells that reference COLAB_ROOT.
COLAB_ROOT = REPO_ROOT

# Ensure `from src.xxx import yyy` works, and run relative to repo root.
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
os.chdir(REPO_ROOT)

# Create the conventional output folders if they don't exist.
for folder in ('logs', 'checkpoints', 'reports'):
    (REPO_ROOT / folder).mkdir(parents=True, exist_ok=True)

# GPU / accelerator summary.
gpus = tf.config.list_physical_devices('GPU')
print(f'GPU devices: {gpus}')
print(f'IS_COLAB={IS_COLAB}  |  Working directory: {REPO_ROOT}')

In [ ]:
# Cell 2: Ultra-Aggressive DQN Configuration (MUST Beat PG's 60.8%)
# Maximum performance parameters to beat PG model
DQN_LEARNING_RATE = 5e-4          # Higher learning rate for faster convergence
DQN_EPSILON_START = 1.0
DQN_EPSILON_END = 0.01            # Very low final epsilon for maximum exploitation
DQN_EPSILON_DECAY_STEPS = 1000    # Very fast decay for quick learning
DQN_BATCH_SIZE = 128              # Large batches for GPU efficiency
DQN_BUFFER_SIZE = 20000            # Large buffer for diverse experiences
DQN_TARGET_UPDATE_FREQ = 25       # Very frequent target updates
DQN_GAMMA = 0.99                  # High gamma for long-term planning

# Maximum training parameters for competitive performance
COLAB_EPISODES = 50               # Much more training
COLAB_GAMES_PER_EPISODE = 30      # More games per episode
COLAB_BUFFER_SIZE = 15000
COLAB_TARGET_UPDATE_FREQ = 20     # Ultra-frequent updates
COLAB_TRAINING_STEPS_PER_EPISODE = 80  # Maximum training steps

# Ultra-aggressive learning rate schedule
def ultra_aggressive_lr_schedule(epoch: int) -> float:
    """Ultra-aggressive learning rate for maximum performance."""
    if epoch < 15:
        return DQN_LEARNING_RATE
    elif epoch < 30:
        return DQN_LEARNING_RATE * 0.7
    elif epoch < 40:
        return DQN_LEARNING_RATE * 0.5
    else:
        return DQN_LEARNING_RATE * 0.3

# Ultra-fast epsilon schedule for rapid exploitation
def ultra_fast_epsilon_schedule(step: int) -> float:
    """Ultra-fast epsilon decay for quick learning."""
    if step < 300:
        return 1.0 * (0.2 ** (step / 150))  # Very fast initial decay
    elif step < 700:
        progress = (step - 300) / 400
        return 0.2 + (0.05 - 0.2) * progress
    else:
        progress = min(1.0, (step - 700) / 300)
        return 0.05 + (DQN_EPSILON_END - 0.05) * progress

print(f"ULTRA-AGGRESSIVE DQN Configuration (MUST Beat PG's 60.8%):")
print(f"  Episodes: {COLAB_EPISODES} (vs PG's 500 groups)")
print(f"  Games per Episode: {COLAB_GAMES_PER_EPISODE}")
print(f"  Total Games: {COLAB_EPISODES * COLAB_GAMES_PER_EPISODE}")
print(f"  Learning Rate: {DQN_LEARNING_RATE}")
print(f"  Epsilon Decay Steps: {DQN_EPSILON_DECAY_STEPS} (very fast)")
print(f"  Batch Size: {DQN_BATCH_SIZE}")
print(f"  Buffer Size: {COLAB_BUFFER_SIZE}")
print(f"  Target Update Freq: {COLAB_TARGET_UPDATE_FREQ}")
print(f"  Training Steps per Episode: {COLAB_TRAINING_STEPS_PER_EPISODE}")
print(f"  Target Performance: >60.8% win rate (BEAT PG)")
print(f"  Expected Training Time: ~{COLAB_EPISODES * 2}-{COLAB_EPISODES * 3} minutes")

In [ ]:
# Cell 3: Enhanced DQN Architecture
def create_enhanced_dqn_model():
    """Enhanced DQN network with Colab optimizations."""
    inputs = layers.Input(shape=(6, 7, 2))
    
    # Enhanced feature extraction
    x = layers.Conv2D(128, (3, 3), activation='relu', padding='same')(inputs)
    x = layers.BatchNormalization()(x)
    x = layers.Conv2D(256, (3, 3), activation='relu', padding='same')(x)
    x = layers.BatchNormalization()(x)
    x = layers.GlobalAveragePooling2D()(x)
    
    # Dense layers with dropout
    x = layers.Dense(256, activation='relu')(x)
    x = layers.Dropout(0.3)(x)
    x = layers.Dense(128, activation='relu')(x)
    x = layers.Dropout(0.2)(x)
    
    # Output layer
    outputs = layers.Dense(7, activation='linear')(x)
    
    model = models.Model(inputs=inputs, outputs=outputs)
    return model

# Test model creation
model = create_enhanced_dqn_model()
print(f"Enhanced DQN model created with {model.count_params():,} parameters")
model.summary()

# Action selection function
def select_action(state: np.ndarray, q_network: tf.keras.Model, 
                 epsilon: float, legal_cols: List[int]) -> Tuple[int, float]:
    """Enhanced epsilon-greedy action selection."""
    if random.random() < epsilon:
        # Explore: random legal action
        action = random.choice(legal_cols)
        q_val = 0.0
    else:
        # Exploit: best legal action
        state_tensor = tf.convert_to_tensor(state[np.newaxis, ...], dtype=tf.float32)
        q_values = q_network(state_tensor, training=False)[0].numpy()
        
        # Mask illegal actions
        masked_q = np.full(7, -np.inf)
        masked_q[legal_cols] = q_values[legal_cols]
        action = int(np.argmax(masked_q))
        q_val = q_values[action]
    
    return action, q_val

print("Enhanced DQN components ready!")

In [ ]:
# Cell 4: Optimized Progressive Opponent System (Performance-Based)
class OptimizedProgressiveOpponent:
    """Performance-based progressive opponent with adaptive difficulty."""
    def __init__(self, strength_level: int = 1):
        self.strength_level = strength_level
        
    def predict_probs(self, board: np.ndarray, player: int) -> np.ndarray:
        """Generate action probabilities based on strength level."""
        legal_cols = [col for col in range(7) if board[0, col] == 0]
        
        if not legal_cols:
            return np.array([1/7] * 7)
        
        if self.strength_level == 1:
            # Level 1: Pure random (easy for learning)
            probs = np.zeros(7)
            probs[legal_cols] = 1.0 / len(legal_cols)
            return probs
            
        elif self.strength_level == 2:
            # Level 2: Slightly strategic
            probs = np.zeros(7)
            center_bias = np.array([0.1, 0.15, 0.25, 0.3, 0.25, 0.15, 0.1])
            for col in legal_cols:
                probs[col] = center_bias[col]
            probs = probs / probs.sum()
            return probs
            
        elif self.strength_level == 3:
            # Level 3: Win-blocking strategy
            probs = np.zeros(7)
            for col in legal_cols:
                if self._check_winning_move(board, col, -player):
                    probs[col] = 3.0
                elif self._check_winning_move(board, col, player):
                    probs[col] = 2.5
                else:
                    probs[col] = 1.0
            if probs.sum() > 0:
                probs = probs / probs.sum()
            else:
                probs[legal_cols] = 1.0 / len(legal_cols)
            return probs
            
        elif self.strength_level == 4:
            # Level 4: Advanced strategy
            probs = np.zeros(7)
            for col in legal_cols:
                # Check for immediate wins
                if self._check_winning_move(board, col, player):
                    probs[col] = 4.0
                elif self._check_winning_move(board, col, -player):
                    probs[col] = 3.5
                # Check for opponent wins to block
                elif self._check_winning_move(board, col, -player):
                    probs[col] = 3.0
                # Center preference
                elif col in [3, 4]:
                    probs[col] = 2.0
                else:
                    probs[col] = 1.0
            if probs.sum() > 0:
                probs = probs / probs.sum()
            else:
                probs[legal_cols] = 1.0 / len(legal_cols)
            return probs
            
        else:
            # Level 5: Expert strategy
            probs = np.zeros(7)
            for col in legal_cols:
                # Immediate win check
                if self._check_winning_move(board, col, player):
                    probs[col] = 5.0
                # Block opponent wins
                elif self._check_winning_move(board, col, -player):
                    probs[col] = 4.5
                # Strategic positioning
                elif col in [2, 3, 4]:
                    probs[col] = 3.0
                elif col in [3, 4]:
                    probs[col] = 2.5
                else:
                    probs[col] = 1.5
            if probs.sum() > 0:
                probs = probs / probs.sum()
            else:
                probs[legal_cols] = 1.0 / len(legal_cols)
            return probs
    
    def _check_winning_move(self, board: np.ndarray, col: int, player: int) -> bool:
        """Check if placing at col would win for player."""
        test_board = board.copy()
        for row in range(5, -1, -1):
            if test_board[row, col] == 0:
                test_board[row, col] = player
                # Check horizontal
                if col <= 3 and all(test_board[row, col+i] == player for i in range(4)):
                    return True
                # Check vertical
                if row <= 2 and all(test_board[row+i, col] == player for i in range(4)):
                    return True
                # Check diagonal (top-left to bottom-right)
                if col <= 3 and row <= 2 and all(test_board[row+i, col+i] == player for i in range(4)):
                    return True
                # Check diagonal (top-right to bottom-left)
                if col >= 3 and row <= 2 and all(test_board[row+i, col-i] == player for i in range(4)):
                    return True
        return False

class AdaptiveOpponentPool:
    """Adaptive opponent pool based on DQN performance."""
    def __init__(self):
        self.current_difficulty = 1
        self.performance_history = []
        
    def get_current_opponent(self, episode: int, dqn_win_rate: float) -> OptimizedProgressiveOpponent:
        """Adaptive difficulty based on DQN performance."""
        # Track performance
        self.performance_history.append(dqn_win_rate)
        recent_performance = self.performance_history[-5:] if len(self.performance_history) >= 5 else self.performance_history
        
        avg_recent = np.mean(recent_performance) if recent_performance else dqn_win_rate
        
        # Adaptive difficulty scaling
        if avg_recent < 0.3:
            # Poor performance - easier opponents
            target_strength = max(1, int(episode // 10))
        elif avg_recent < 0.5:
            # Average performance - moderate opponents
            target_strength = max(2, int(episode // 8))
        elif avg_recent < 0.7:
            # Good performance - harder opponents
            target_strength = max(3, int(episode // 6))
        else:
            # Excellent performance - expert opponents
            target_strength = max(4, int(episode // 4))
        
        # Gradual difficulty increase
        self.current_difficulty = min(5, target_strength)
        
        return OptimizedProgressiveOpponent(self.current_difficulty)

# Optimized Mock Game Engine for faster execution
class OptimizedMockGameEngine:
    def __init__(self):
        self.move_cache = {}  # Cache move results
        
    def legal_moves(self, board):
        return [col for col in range(7) if board[0, col] == 0]
    
    def make_move(self, board, col, player):
        new_board = board.copy()
        for row in range(5, -1, -1):
            if new_board[row, col] == 0:
                new_board[row, col] = player
                break
        return new_board
    
    def check_win(self, board, player):
        cache_key = (tuple(board.flatten()), player)
        if cache_key in self.move_cache:
            return self.move_cache[cache_key]
        
        # Check horizontal
        for row in range(6):
            for col in range(4):
                if all(board[row, col+i] == player for i in range(4)):
                    self.move_cache[cache_key] = True
                    return True
        
        # Check vertical
        for col in range(7):
            for row in range(3):
                if all(board[row+i, col] == player for i in range(4)):
                    self.move_cache[cache_key] = True
                    return True
        
        # Check diagonals
        for row in range(3):
            for col in range(4):
                if all(board[row+i, col+i] == player for i in range(4)):
                    self.move_cache[cache_key] = True
                    return True
                if all(board[row+i, col-i] == player for i in range(4)):
                    self.move_cache[cache_key] = True
                    return True
        
        self.move_cache[cache_key] = False
        return False
    
    def random_moves(self, board, num_moves):
        current_board = board.copy()
        current_player = 1
        
        for _ in range(num_moves):
            legal = self.legal_moves(current_board)
            if not legal:
                return current_board, None
            
            # Smart random moves - avoid obvious mistakes
            if len(legal) > 1:
                # Avoid giving opponent immediate win
                for col in legal:
                    test_board = self.make_move(current_board, col, current_player)
                    if self.check_win(test_board, current_player):
                        legal.remove(col)
                        break
            
            col = random.choice(legal) if legal else 0
            current_board = self.make_move(current_board, col, current_player)
            
            if self.check_win(current_board, current_player):
                return current_board, None
            
            current_player = -current_player
        
        return current_board, current_player

# Initialize optimized components
game_engine = OptimizedMockGameEngine()
adaptive_pool = AdaptiveOpponentPool()

print("Optimized Progressive Opponent System Ready!")
print(f"  - Adaptive difficulty based on DQN performance")
print(f"  - 5 strength levels with strategic behavior")
print(f"  - Caching for faster game simulation")
print(f"  - Smart random initialization")

In [ ]:
# Cell 5: Optimized Game Functions (Faster Execution)
def play_optimized_dqn_game(dqn_model, opponent, epsilon, random_init_moves=4):
    """Optimized DQN vs opponent game with performance tracking."""
    experiences = []
    
    # Initialize empty board
    board = np.zeros((6, 7), dtype=np.int8)
    
    # Randomly assign who goes first
    dqn_player = random.choice([+1, -1])
    opponent_player = -dqn_player
    
    # Apply optimized random warm-up moves
    board, next_player = game_engine.random_moves(board, random_init_moves)
    if next_player is None:
        return experiences
    
    # Main game loop with performance optimization
    current_player = next_player
    done = False
    winner = None
    move_count = 0
    
    while not done and move_count < 42:  # Limit moves for faster execution
        if current_player == dqn_player:
            # DQN's turn - optimized
            legal_cols = game_engine.legal_moves(board)
            if not legal_cols:
                break
            
            # Encode board for DQN
            state = encode_board_enhanced(board, dqn_player)
            
            # Select action with optimized epsilon-greedy
            action, q_val = select_action(state, dqn_model, epsilon, legal_cols)
            
            # Make move
            board = game_engine.make_move(board, action, dqn_player)
            move_count += 1
            
            # Quick win check
            if game_engine.check_win(board, dqn_player):
                done = True
                winner = dqn_player
            elif len(game_engine.legal_moves(board)) == 0:
                done = True
                winner = None
            else:
                current_player = opponent_player
        else:
            # Optimized opponent's turn
            legal_cols = game_engine.legal_moves(board)
            if not legal_cols:
                break
            
            # Use optimized opponent with caching
            opp_probs = opponent.predict_probs(board, opponent_player)
            action = np.random.choice(7, p=opp_probs)
            
            # Make move
            board = game_engine.make_move(board, action, opponent_player)
            move_count += 1
            
            # Quick win check
            if game_engine.check_win(board, opponent_player):
                done = True
                winner = opponent_player
            elif len(game_engine.legal_moves(board)) == 0:
                done = True
                winner = None
            else:
                current_player = dqn_player
        
        # Calculate reward and next state (optimized)
        if done:
            if winner == dqn_player:
                reward = 1.0
            elif winner == opponent_player:
                reward = -1.0
            else:
                reward = 0.0
            
            # Next state for DQN's next turn
            if current_player == dqn_player and not done:
                next_state = encode_board_enhanced(board, dqn_player)
            else:
                next_state = np.zeros_like(state)
            
            experiences.append((state, action, reward, next_state, done))
    
    return experiences

# Adaptive learning rate based on performance
def adaptive_learning_rate_schedule(episode: int, current_win_rate: float) -> float:
    """Adaptive learning rate based on current performance."""
    base_lr = DQN_LEARNING_RATE
    
    # Adjust based on performance
    if current_win_rate < 0.3:
        # Poor performance - increase learning rate
        return base_lr * 1.5
    elif current_win_rate < 0.5:
        # Average performance - standard learning rate
        return base_lr
    elif current_win_rate < 0.7:
        # Good performance - decrease learning rate
        return base_lr * 0.7
    else:
        # Excellent performance - significant decrease
        return base_lr * 0.5

print("Optimized Game Functions Ready!")
print("  - Faster game execution with move limits")
print("  - Adaptive learning rate based on performance")
print("  - Optimized opponent caching")

In [ ]:
# Cell 6: Enhanced DQN Training (Colab Optimized)
def encode_board_enhanced(board, player):
    """Enhanced board encoding."""
    encoded = np.zeros((6, 7, 2), dtype=np.float32)
    encoded[:, :, 0] = (board == player).astype(np.float32)
    encoded[:, :, 1] = (board == -player).astype(np.float32)
    return encoded

def play_enhanced_dqn_game(dqn_model, opponent, epsilon, random_init_moves=4):
    """Play DQN vs opponent with experience collection."""
    experiences = []
    
    # Initialize empty board
    board = np.zeros((6, 7), dtype=np.int8)
    
    # Randomly assign who goes first
    dqn_player = random.choice([+1, -1])
    opponent_player = -dqn_player
    
    # Apply random warm-up moves
    board, next_player = game_engine.random_moves(board, random_init_moves)
    if next_player is None:
        return experiences
    
    # Main game loop
    current_player = next_player
    done = False
    winner = None
    
    while not done:
        if current_player == dqn_player:
            # DQN's turn
            legal_cols = game_engine.legal_moves(board)
            if not legal_cols:
                break
            
            # Encode board for DQN
            state = encode_board_enhanced(board, dqn_player)
            
            # Select action
            action, q_val = select_action(state, dqn_model, epsilon, legal_cols)
            
            # Make move
            board = game_engine.make_move(board, action, dqn_player)
            
            # Check if game ended
            if len(game_engine.legal_moves(board)) == 0:
                done = True
                winner = None
            elif game_engine.check_win(board, dqn_player):
                done = True
                winner = dqn_player
            elif game_engine.check_win(board, opponent_player):
                done = True
                winner = opponent_player
            
            if not done:
                # Opponent's turn
                next_board = board.copy()
                
                # Opponent move
                opp_legal = game_engine.legal_moves(next_board)
                if opp_legal:
                    opp_probs = opponent.predict_probs(next_board, opponent_player)
                    opp_action = np.random.choice(7, p=opp_probs)
                    next_board = game_engine.make_move(next_board, opp_action, opponent_player)
                
                # Check if game ended after opponent move
                if len(game_engine.legal_moves(next_board)) == 0:
                    done = True
                    winner = None
                elif game_engine.check_win(next_board, dqn_player):
                    done = True
                    winner = dqn_player
                elif game_engine.check_win(next_board, opponent_player):
                    done = True
                    winner = opponent_player
                
                # Next state setup
                if not done:
                    next_state = encode_board_enhanced(next_board, dqn_player)
                else:
                    next_state = np.zeros_like(state)
                
                # Calculate reward
                if winner == dqn_player:
                    reward = 1.0
                elif winner == opponent_player:
                    reward = -1.0
                else:
                    reward = 0.0
                
                experiences.append((state, action, reward, next_state, done))
                board = next_board
            else:
                # Game ended on DQN's move
                if winner == dqn_player:
                    reward = 1.0
                else:
                    reward = -1.0
                
                next_state = np.zeros_like(state)
                experiences.append((state, action, reward, next_state, done))
        else:
            break
    
    return experiences

print("Enhanced DQN game engine ready!")

In [ ]:
# Cell 7: Training Execution (Colab Optimized)
print("Starting Enhanced DQN Training on Colab...")
print("=" * 50)

# Create enhanced networks
q_network = create_enhanced_dqn_model()
target_network = create_enhanced_dqn_model()
target_network.set_weights(q_network.get_weights())

# Optimizer
optimizer = optimizers.Adam(learning_rate=DQN_LEARNING_RATE)

# Training metrics
training_log = {
    "episode": [],
    "loss": [],
    "win_rate": [],
    "epsilon": [],
    "buffer_size": [],
    "episode_time": [],
    "learning_rate": [],
}

# Simple replay buffer (Colab optimized)
from collections import deque
replay_buffer = deque(maxlen=COLAB_BUFFER_SIZE)

total_steps = 0
recent_wins = deque(maxlen=50)

print(f"Training Configuration:")
print(f"  Episodes: {COLAB_EPISODES}")
print(f"  Games per Episode: {COLAB_GAMES_PER_EPISODE}")
print(f"  Buffer Size: {COLAB_BUFFER_SIZE}")
print(f"  Target Update Freq: {COLAB_TARGET_UPDATE_FREQ}")
print(f"  Total Games: {COLAB_EPISODES * COLAB_GAMES_PER_EPISODE}")
print(f"  GPU: {'Enabled' if tf.config.list_physical_devices('GPU') else 'CPU Only'}")

In [ ]:
# Cell 8: Optimized Training Loop (Maximum Performance)
start_time = time.time()

print("Starting OPTIMIZED Enhanced DQN Training on GPU...")
print("TARGET: BEAT PG MODEL'S 60.8% AVERAGE WIN RATE")
print("=" * 70)

for episode in range(COLAB_EPISODES):
    episode_start_time = time.time()
    
    # Get adaptive opponent based on performance
    current_win_rate = np.mean(recent_wins) if recent_wins else 0.0
    opponent = adaptive_pool.get_current_opponent(episode, current_win_rate)
    
    # Calculate current epsilon with ultra-fast schedule
    epsilon = ultra_fast_epsilon_schedule(total_steps)
    
    # Calculate adaptive learning rate based on performance
    current_lr = adaptive_learning_rate_schedule(episode, current_win_rate)
    optimizer.learning_rate.assign(current_lr)
    
    # Play optimized games for this episode
    episode_experiences = []
    wins = 0
    
    for game in range(COLAB_GAMES_PER_EPISODE):
        experiences = play_optimized_dqn_game(q_network, opponent, epsilon)
        episode_experiences.extend(experiences)
        
        # Count wins
        if experiences and experiences[-1][2] == 1.0:
            wins += 1
            recent_wins.append(1)
        elif experiences:
            recent_wins.append(0)
    
    # Add experiences to replay buffer
    for exp in episode_experiences:
        replay_buffer.append(exp)
    
    # Optimized training on replay buffer
    episode_losses = []
    if len(replay_buffer) >= DQN_BATCH_SIZE:
        # Maximum training steps per episode
        num_training_steps = min(COLAB_TRAINING_STEPS_PER_EPISODE, len(episode_experiences))
        
        for step in range(num_training_steps):
            # Prioritized sampling with performance weighting
            if len(replay_buffer) >= DQN_BATCH_SIZE * 2:
                recent_batch_size = DQN_BATCH_SIZE // 2
                older_batch_size = DQN_BATCH_SIZE - recent_batch_size
                
                recent_exp = random.sample(list(replay_buffer)[-DQN_BATCH_SIZE:], min(recent_batch_size, len(replay_buffer)))
                older_exp = random.sample(list(replay_buffer)[:-DQN_BATCH_SIZE], older_batch_size)
                batch = recent_exp + older_exp
            else:
                batch = random.sample(list(replay_buffer), DQN_BATCH_SIZE)
            
            states = np.array([exp[0] for exp in batch])
            actions = np.array([exp[1] for exp in batch], dtype=np.int32)
            rewards = np.array([exp[2] for exp in batch])
            next_states = np.array([exp[3] for exp in batch])
            dones = np.array([exp[4] for exp in batch])
            
            # Enhanced training step with all optimizations
            with tf.GradientTape() as tape:
                # Current Q-values
                current_q = q_network(states, training=True)
                current_q_values = tf.gather_nd(current_q, 
                                               tf.stack([tf.range(len(actions), dtype=tf.int32), actions], axis=1))
                
                # Double DQN with improved targets
                next_q_main = q_network(next_states, training=False)
                next_actions = tf.argmax(next_q_main, axis=1)
                next_q_target = target_network(next_states, training=False)
                
                next_actions_int32 = tf.cast(next_actions, tf.int32)
                max_next_q = tf.gather_nd(next_q_target, 
                                         tf.stack([tf.range(len(next_actions_int32), dtype=tf.int32), next_actions_int32], axis=1))
                
                target_q = rewards + DQN_GAMMA * max_next_q * (1 - dones)
                
                # Enhanced Huber loss
                error = current_q_values - target_q
                loss = tf.where(tf.abs(error) < 1.0, 0.5 * tf.square(error), tf.abs(error) - 0.5)
                loss = tf.reduce_mean(loss)
            
            # Gradient clipping and application
            gradients = tape.gradient(loss, q_network.trainable_variables)
            clipped_gradients = [tf.clip_by_norm(grad, 0.5) for grad in gradients]  # Tighter clipping
            optimizer.apply_gradients(zip(clipped_gradients, q_network.trainable_variables))
            
            episode_losses.append(loss.numpy())
            total_steps += 1
    
    # Ultra-frequent target network updates
    if episode % COLAB_TARGET_UPDATE_FREQ == 0:
        target_network.set_weights(q_network.get_weights())
    
    # Calculate metrics
    avg_loss = np.mean(episode_losses) if episode_losses else 0.0
    win_rate = wins / COLAB_GAMES_PER_EPISODE
    episode_time = time.time() - episode_start_time
    
    # Log metrics
    training_log["episode"].append(episode)
    training_log["loss"].append(float(avg_loss))
    training_log["win_rate"].append(float(win_rate))
    training_log["epsilon"].append(float(epsilon))
    training_log["buffer_size"].append(len(replay_buffer))
    training_log["episode_time"].append(float(episode_time))
    training_log["learning_rate"].append(float(current_lr))
    
    # Enhanced progress reporting with performance tracking
    current_avg = np.mean(training_log["win_rate"][-5:]) if len(training_log["win_rate"]) >= 5 else win_rate
    pg_comparison = "WINNING" if current_avg > 0.608 else "CATCHING UP" if current_avg > 0.4 else "BEHIND"
    
    if episode % 5 == 0 or episode == COLAB_EPISODES - 1:
        print(f"Episode {episode:2d} | Loss: {avg_loss:.4f} | Win Rate: {win_rate:.3f} | "
              f"Epsilon: {epsilon:.3f} | LR: {current_lr:.6f} | Buffer: {len(replay_buffer)} | "
              f"Time: {episode_time:.2f}s | Steps: {total_steps} | STATUS: {pg_comparison}")
    else:
        print(f"Episode {episode:2d} | Win Rate: {win_rate:.3f} | Epsilon: {epsilon:.3f} | "
              f"STATUS: {pg_comparison} | Time: {episode_time:.2f}s")

total_time = time.time() - start_time
total_games = COLAB_EPISODES * COLAB_GAMES_PER_EPISODE

print(f"\nOPTIMIZED Enhanced DQN Training completed in {total_time:.2f}s")
print(f"Total games played: {total_games}")
print(f"Average games per second: {total_games/total_time:.2f}")
print(f"Final win rate: {training_log['win_rate'][-1]:.3f}")
print(f"Final epsilon: {training_log['epsilon'][-1]:.3f}")
print(f"Final loss: {training_log['loss'][-1]:.4f}")
print(f"Average episode time: {np.mean(training_log['episode_time']):.2f}s")

# Performance summary with PG comparison
best_win_rate = max(training_log['win_rate'])
avg_win_rate = np.mean(training_log['win_rate'])
pg_target = 0.608

print(f"\nPERFORMANCE SUMMARY:")
print(f"  Best Win Rate: {best_win_rate:.3f} ({best_win_rate:.1%})")
print(f"  Average Win Rate: {avg_win_rate:.3f} ({avg_win_rate:.1%})")
print(f"  PG Target: {pg_target:.3f} ({pg_target:.1%})")
print(f"  vs PG Average: {avg_win_rate - pg_target:+.3f} ({(avg_win_rate/pg_target - 1)*100:+.1f}%)")

# Performance assessment
if avg_win_rate > pg_target:
    print(f"\nSUCCESS: OPTIMIZED DQN BEATS PG model! Ready for tournament!")
elif avg_win_rate > 0.5:
    print(f"\nCLOSE: DQN approaching PG performance. Good progress!")
else:
    print(f"\nNEEDS WORK: DQN below target. Consider more training.")

# Save optimized model and logs
model_path = COLAB_ROOT / 'checkpoints' / 'enhanced_dqn_optimized.h5'
log_path = COLAB_ROOT / 'logs' / 'enhanced_dqn_optimized_log.json'

q_network.save(str(model_path))
with open(log_path, 'w') as f:
    json.dump(training_log, f, indent=2)

print(f"\nOptimized model saved to: {model_path}")
print(f"Training logs saved to: {log_path}")

print(f"\nOPTIMIZED Enhanced DQN training completed!")
print(f"Final Status: {'BEATS PG' if avg_win_rate > pg_target else 'NEEDS IMPROVEMENT'}")

In [ ]:
# Cell 9: Performance Comparison & Visualization (Fixed JSON + Graphs)
print("Performance Comparison: Enhanced DQN vs PG Model")
print("=" * 60)

# PG Model Baseline Performance (from training log)
PG_PERFORMANCE = {
    'average_win_rate': 0.608,
    'best_win_rate': 0.950,
    'final_win_rate': 0.550,
    'training_groups': 500
}

# Enhanced DQN Performance
dqn_final_win_rate = training_log['win_rate'][-1]
dqn_best_win_rate = max(training_log['win_rate'])
dqn_avg_win_rate = np.mean(training_log['win_rate'])
dqn_improvement = dqn_final_win_rate - training_log['win_rate'][0]

print(f"PG Model Baseline:")
print(f"  Average Win Rate: {PG_PERFORMANCE['average_win_rate']:.1%}")
print(f"  Best Win Rate: {PG_PERFORMANCE['best_win_rate']:.1%}")
print(f"  Final Win Rate: {PG_PERFORMANCE['final_win_rate']:.1%}")
print(f"  Training Groups: {PG_PERFORMANCE['training_groups']}")

print(f"\nEnhanced DQN Performance:")
print(f"  Average Win Rate: {dqn_avg_win_rate:.1%}")
print(f"  Best Win Rate: {dqn_best_win_rate:.1%}")
print(f"  Final Win Rate: {dqn_final_win_rate:.1%}")
print(f"  Episodes: {len(training_log['episode'])}")
print(f"  Total Games: {COLAB_EPISODES * COLAB_GAMES_PER_EPISODE}")

print(f"\nPerformance Comparison:")
print(f"  Average: {dqn_avg_win_rate:.1%} vs {PG_PERFORMANCE['average_win_rate']:.1%} PG")
print(f"  Best: {dqn_best_win_rate:.1%} vs {PG_PERFORMANCE['best_win_rate']:.1%} PG")
print(f"  Final: {dqn_final_win_rate:.1%} vs {PG_PERFORMANCE['final_win_rate']:.1%} PG")

# Determine if DQN beats PG
avg_comparison = "BEATS" if dqn_avg_win_rate > PG_PERFORMANCE['average_win_rate'] else "MATCHES" if abs(dqn_avg_win_rate - PG_PERFORMANCE['average_win_rate']) < 0.05 else "BELOW"
best_comparison = "BEATS" if dqn_best_win_rate > PG_PERFORMANCE['best_win_rate'] else "MATCHES" if abs(dqn_best_win_rate - PG_PERFORMANCE['best_win_rate']) < 0.05 else "BELOW"
final_comparison = "BEATS" if dqn_final_win_rate > PG_PERFORMANCE['final_win_rate'] else "MATCHES" if abs(dqn_final_win_rate - PG_PERFORMANCE['final_win_rate']) < 0.05 else "BELOW"

print(f"\nResult Summary:")
print(f"  Average Performance: {avg_comparison} PG baseline")
print(f"  Peak Performance: {best_comparison} PG baseline")
print(f"  Final Performance: {final_comparison} PG baseline")

# Overall assessment
if dqn_avg_win_rate >= PG_PERFORMANCE['average_win_rate']:
    print(f"\nSUCCESS: Enhanced DQN {avg_comparison} PG model performance!")
    print(f"Ready for tournament with competitive performance.")
else:
    print(f"\nNEEDS IMPROVEMENT: DQN below PG average performance.")
    print(f"Consider: More episodes, different hyperparameters, or longer training.")

# Performance improvement over training
if len(training_log['win_rate']) > 1:
    improvement = dqn_final_win_rate - training_log['win_rate'][0]
    print(f"\nTraining Improvement: {improvement:+.1%} ({improvement*100:+.1f} percentage points)")

# Create performance graphs
import matplotlib.pyplot as plt

# Create figure with subplots
fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2, 2, figsize=(15, 12))
fig.suptitle('Enhanced DQN Performance Analysis', fontsize=16, fontweight='bold')

# Plot 1: Win Rate Progress
ax1.plot(training_log['episode'], training_log['win_rate'], 'b-', linewidth=2, label='DQN Win Rate')
ax1.axhline(y=PG_PERFORMANCE['average_win_rate'], color='r', linestyle='--', label=f'PG Average: {PG_PERFORMANCE["average_win_rate"]:.1%}')
ax1.set_title('Win Rate Progression', fontweight='bold')
ax1.set_xlabel('Episode')
ax1.set_ylabel('Win Rate')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Plot 2: Loss Progress
ax2.plot(training_log['episode'], training_log['loss'], 'g-', linewidth=2, label='DQN Loss')
ax2.set_title('Training Loss', fontweight='bold')
ax2.set_xlabel('Episode')
ax2.set_ylabel('Loss')
ax2.legend()
ax2.grid(True, alpha=0.3)

# Plot 3: Epsilon Decay
ax3.plot(training_log['episode'], training_log['epsilon'], 'r-', linewidth=2, label='Epsilon')
ax3.set_title('Epsilon Decay Schedule', fontweight='bold')
ax3.set_xlabel('Episode')
ax3.set_ylabel('Epsilon')
ax3.legend()
ax3.grid(True, alpha=0.3)

# Plot 4: Learning Rate Schedule
ax4.plot(training_log['episode'], training_log['learning_rate'], 'm-', linewidth=2, label='Learning Rate')
ax4.set_title('Learning Rate Schedule', fontweight='bold')
ax4.set_xlabel('Episode')
ax4.set_ylabel('Learning Rate')
ax4.legend()
ax4.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Performance comparison bar chart
fig, ax = plt.subplots(figsize=(10, 6))

metrics = ['Average', 'Best', 'Final']
dqn_values = [dqn_avg_win_rate, dqn_best_win_rate, dqn_final_win_rate]
pg_values = [PG_PERFORMANCE['average_win_rate'], PG_PERFORMANCE['best_win_rate'], PG_PERFORMANCE['final_win_rate']]

x = np.arange(len(metrics))
width = 0.35

bars1 = ax.bar(x - width/2, dqn_values, width, label='Enhanced DQN', color='blue', alpha=0.7)
bars2 = ax.bar(x + width/2, pg_values, width, label='PG Model', color='red', alpha=0.7)

ax.set_title('DQN vs PG Performance Comparison', fontweight='bold', fontsize=14)
ax.set_xlabel('Performance Metric')
ax.set_ylabel('Win Rate')
ax.set_xticks(x)
ax.set_xticklabels(metrics)
ax.legend()
ax.grid(True, alpha=0.3)

# Add value labels on bars
for bars in [bars1, bars2]:
    for bar in bars:
        height = bar.get_height()
        ax.annotate(f'{height:.1%}',
                    xy=(bar.get_x() + bar.get_width() / 2, height),
                    xytext=(0, 3),  # 3 points vertical offset
                    textcoords="offset points",
                    ha='center', va='bottom')

plt.tight_layout()
plt.show()

# Create reports directory if it doesn't exist
reports_dir = COLAB_ROOT / 'reports'
reports_dir.mkdir(exist_ok=True)

# Save performance comparison (fixed JSON serialization)
performance_comparison = {
    'pg_baseline': PG_PERFORMANCE,
    'dqn_performance': {
        'average_win_rate': float(dqn_avg_win_rate),
        'best_win_rate': float(dqn_best_win_rate),
        'final_win_rate': float(dqn_final_win_rate),
        'episodes': len(training_log['episode']),
        'total_games': COLAB_EPISODES * COLAB_GAMES_PER_EPISODE,
        'training_improvement': float(dqn_final_win_rate - training_log['win_rate'][0] if len(training_log['win_rate']) > 1 else 0)
    },
    'comparison_result': {
        'average_comparison': avg_comparison,
        'best_comparison': best_comparison,
        'final_comparison': final_comparison,
        'overall_success': bool(dqn_avg_win_rate >= PG_PERFORMANCE['average_win_rate'])
    },
    'timestamp': time.strftime('%Y-%m-%d %H:%M:%S')
}

comparison_path = reports_dir / 'dqn_vs_pg_comparison.json'
with open(comparison_path, 'w') as f:
    json.dump(performance_comparison, f, indent=2)

print(f"\nPerformance comparison saved to: {comparison_path}")
print(f"Tournament Status: {'READY' if dqn_avg_win_rate >= PG_PERFORMANCE['average_win_rate'] else 'NEEDS MORE TRAINING'}")

# Summary statistics
print(f"\n📊 Final Summary:")
print(f"🏆 DQN Average: {dqn_avg_win_rate:.1%} vs PG: {PG_PERFORMANCE['average_win_rate']:.1%}")
print(f"🎯 Peak Performance: {dqn_best_win_rate:.1%} vs PG: {PG_PERFORMANCE['best_win_rate']:.1%}")
print(f"📈 Final Performance: {dqn_final_win_rate:.1%} vs PG: {PG_PERFORMANCE['final_win_rate']:.1%}")
print(f"✅ Overall Result: {avg_comparison} PG baseline")